In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.colors as mcolors
import seaborn as sns
import json
import numpy as np
from matplotlib.ticker import FixedLocator, ScalarFormatter
import math
import os

In [ ]:
DIR_PATH = "./../data/datasets/manually_cleaned/approved_manually_cleaned/"
EMA_path = DIR_PATH + "EMA.csv"
SWISSMEDIC_path = DIR_PATH + "Swissmedic.csv"
AUSTRALIA_path = DIR_PATH + "TGA.csv"
FDA_path = DIR_PATH + "FDA.csv"
HEALTHCANADA_path = DIR_PATH + "HealthCanada.csv"

# NOTE: PMDA / Japan is intentionally excluded from this analysis.

In [ ]:
# CSV loader
def load_agency_csv(path: str, agency: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.astype(str).str.strip()
    return df

# Load all agencies
df_ema = load_agency_csv(EMA_path, "EMA")
# df_fda = load_agency_csv(FDA_path, "FDA")
df_swissmedic = load_agency_csv(SWISSMEDIC_path, "SWISSMEDIC")
df_australia = load_agency_csv(AUSTRALIA_path, "AUSTRALIA")
# df_healthcanada = load_agency_csv(HEALTHCANADA_path, "HEALTHCANADA")

# Identifier counting helpers
PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def cleaned_series(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    x = x.mask(x.str.lower().isin(PLACEHOLDERS))
    return x

def agency_identifier_count(df: pd.DataFrame, agency: str) -> int:
    """
    Count the agency-specific identifier:
    unique Marketing_authorisation_number values without placeholders.
    """
    s = cleaned_series(df["Marketing_authorisation_number"])
    return s.nunique(dropna=True)

# Results 1. Dataset Characteristics

Numbers per application

In [ ]:
agencies = {
    # "FDA": df_fda,
    # "Health Canada": df_healthcanada,
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "Australia": df_australia,
}

# 1) Record counts per agency (based on CSV rows)
rows = []
total_records = sum(len(df) for df in agencies.values())

for name, df in agencies.items():
    n = len(df)
    pct = round(n / total_records * 100, 2) if total_records else 0.0
    rows.append({
        "Agency": name,
        "n_records": n,
        "%_of_overall_records": pct
    })

record_summary = pd.DataFrame(rows)

print("Record counts per agency (based on CSV rows)")
display(record_summary)

# Number per Approval (Consolidated from Approved, Conditional Marketing Authorisation, Marketed)

In [ ]:
# Approved applications per agency (record counts, no normalisation)

APPROVAL_DECISIONS = {
    "approved",
    "conditional marketing authorisation",
    "marketed",
}

rows = []
total_records_approved = 0

for name, df in agencies.items():
    n_approved = int(df["Decision"].isin(APPROVAL_DECISIONS).sum())

    rows.append({
        "Agency": name,
        "n_records_approved": n_approved,
    })

    total_records_approved += n_approved

approved_record_summary = pd.DataFrame(rows)

approved_record_summary["%_of_overall_approved_records"] = (
    approved_record_summary["n_records_approved"] / total_records_approved * 100
).round(2)

print("Approved applications per agency (record counts, no normalisation)")
display(approved_record_summary)

# Number per drug (unique Marketing_authorisation_number)

In [ ]:
# Unique identifiers per agency (MA numbers)
def clean_ma_series(s: pd.Series) -> set:
    return set(
        s.astype("string")
         .str.strip()
         .str.lower()
         .mask(lambda x: x.isin(PLACEHOLDERS))
         .dropna()
         .unique()
)

agency_ids = {}
overall_ids = 0

for name, df in agencies.items():
    ma_set = clean_ma_series(df["Marketing_authorisation_number"])
    n_ids = len(ma_set)
    agency_ids[name] = n_ids
    overall_ids += n_ids

rows = []
for name, n in agency_ids.items():
    pct = round(n / overall_ids * 100, 2) if overall_ids else 0.0
    rows.append({
        "Agency": name,
        "n_unique_identifiers": n,
        "%_of_overall": pct
    })

ma_summary = pd.DataFrame(rows)

print("Unique identifiers per agency (MA numbers)")
display(ma_summary)

# Study Flow Chart

In [ ]:
# Paths
BASE_MC = Path("./../data/datasets/manually_cleaned")
BASE_1995 = Path("./../data/datasets/1995")

ALL_DECISIONS = BASE_1995 / "all_decisions"
APPROVED = BASE_1995 / "approved"

# Agency mapping
# key   = canonical agency name used in tables and the flowchart
# value = filename stem in manually_cleaned
AGENCY_MAPPING = {
    "EMA": "EMA",
    "FDA": "FDA",
    "Health Canada": "HEALTHCANADA",
    "TGA": "AUSTRALIA",
    "Swissmedic": "SWISSMEDIC",
}

PAR_AGENCIES = {"EMA", "TGA", "Swissmedic"}
NON_PAR_AGENCIES = {"FDA", "Health Canada"}

# Helpers
def count_manually_cleaned_json(path):
    """Count entries in manually cleaned JSON (dict-of-dicts)."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return sum(1 for v in data.values() if isinstance(v, dict))

def count_csv(path):
    """Count rows in a CSV."""
    return len(pd.read_csv(path))

def get_docname_and_decision_date_from_json(path):
    """Return {document_name: decision_date}."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}
    for v in data.values():
        if isinstance(v, dict):
            doc = v.get("document_name")
            date = v.get("decision_date")
            if doc:
                out[doc] = date

    return out

def get_ids_from_csv(path):
    """Return the set of ids contained in a CSV."""
    df = pd.read_csv(path)
    return set(df["id"])

# Per-agency sequential counts
rows = []

for agency, mc_stem in AGENCY_MAPPING.items():
    mc_file = BASE_MC / f"{mc_stem}_manually_cleaned.json"
    all_file = ALL_DECISIONS / f"{agency.replace(' ', '')}.csv"
    approved_file = APPROVED / f"{agency.replace(' ', '')}.csv"

    n_retrieved = count_manually_cleaned_json(mc_file)
    n_all_1995 = count_csv(all_file)
    n_approved = count_csv(approved_file)

    rows.append({
        "Agency": agency,
        "Retrieved (manually cleaned)": n_retrieved,
        "Excluded (<1995)": n_retrieved - n_all_1995,
        "Evaluated (>=1995)": n_all_1995,
        "Excluded (not approved)": n_all_1995 - n_approved,
        "Approved": n_approved,
    })

per_agency_df = pd.DataFrame(rows)

# Group-level summary
group_rows = []

for name, agency_set in {
    "PAR agencies": PAR_AGENCIES,
    "FDA + Health Canada": NON_PAR_AGENCIES,
}.items():
    subset = per_agency_df[per_agency_df["Agency"].isin(agency_set)]

    group_rows.append({
        "Group": name,
        "Retrieved": subset["Retrieved (manually cleaned)"].sum(),
        "Excluded (<1995)": subset["Excluded (<1995)"].sum(),
        "Evaluated (>=1995)": subset["Evaluated (>=1995)"].sum(),
        "Excluded (not approved)": subset["Excluded (not approved)"].sum(),
        "Approved": subset["Approved"].sum(),
    })

group_df = pd.DataFrame(group_rows)

print("\n=== Per-agency flowchart numbers ===")
display(per_agency_df)

print("\n=== Group-level flowchart numbers ===")
display(group_df)

# Debug: manually_cleaned.csv vs all_decisions.csv
print("\n=== DEBUG: PAR exclusions using manually cleaned CSVs ===")

debug_rows = []

for agency in PAR_AGENCIES:
    mc_stem = AGENCY_MAPPING[agency]

    mc_csv = BASE_MC / f"{mc_stem}_manually_cleaned.csv"
    all_csv = ALL_DECISIONS / f"{agency.replace(' ', '')}.csv"

    df_mc = pd.read_csv(mc_csv)
    df_all = pd.read_csv(all_csv)

    evaluable_docs = set(df_all["Document_name"])

    for _, row in df_mc.iterrows():
        doc = row.get("Document_name")

        if pd.notna(doc) and doc not in evaluable_docs:
            debug_rows.append({
                "Agency": agency,
                "Document_name": doc,
                "Decision_year": row.get("Decision_year"),
                "Decision_date": row.get("Decision_date"),
                "Decision": row.get("Decision"),
                "Current_status": row.get("Current_status"),
            })

debug_df = pd.DataFrame(debug_rows)
display(debug_df)

debug_df[["Decision_year", "Decision_date"]].apply(pd.unique)

# Table 1: Key characteristics of approvals

In [ ]:
# Constants and helpers (analysis)
DRUG_CLASS_ORDER = [
    "small molecule",
    "biologics",
    "cell and gene therapy",
    "peptides and proteins",
    "vaccine",
    "not reported",
    "other",
]

agencies = {
    "EMA": pd.read_csv(EMA_path),
    "Swissmedic": pd.read_csv(SWISSMEDIC_path),
    "TGA": pd.read_csv(AUSTRALIA_path),
    "FDA": pd.read_csv(FDA_path),
    "Health Canada": pd.read_csv(HEALTHCANADA_path),
}

NON_PAR_AGENCIES = {"FDA", "Health Canada"}
PAR_AGENCIES = {"EMA", "Swissmedic", "TGA"}

def norm_series(s: pd.Series) -> pd.Series:
    """Strip whitespace and lower-case while keeping NaN."""
    return s.astype("string").str.strip().str.lower()

def clean_split_to_unique_list(value):
    """Split ';'-separated disease classes, clean them, and remove placeholders."""
    if pd.isna(value):
        return []

    parts = [p.strip().lower() for p in str(value).split(";")]
    parts = [p for p in parts if p and p not in PLACEHOLDERS]

    return list(dict.fromkeys(parts))

def decision_distribution(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)

    decision = (
        df.get("Decision", pd.Series([pd.NA] * n))
          .astype("string")
          .str.strip()
          .str.lower()
    )

    decision = decision.mask(decision.isin(PLACEHOLDERS), pd.NA)

    counts = decision.value_counts(dropna=False)
    dist = counts.reset_index()
    dist.columns = ["Decision", "n"]
    dist["%"] = (dist["n"] / n * 100).round(2) if n else 0.0

    consolidated_mask = decision.isin({
        "approved",
        "conditional marketing authorisation",
        "conditional marketing authorization",
    })

    consolidated_row = pd.DataFrame([{
        "Decision": "consolidated (approved + conditional marketing authorisation)",
        "n": int(consolidated_mask.sum()),
        "%": round(consolidated_mask.sum() / n * 100, 2) if n else 0.0
    }])

    return pd.concat([dist, consolidated_row], ignore_index=True)

def bucket_drug_class(x) -> str:
    if pd.isna(x):
        return "not reported"

    t = str(x).strip().lower()
    if t in PLACEHOLDERS:
        return "not reported"

    if t in DRUG_CLASS_ORDER:
        return t

    return "other"

def drug_class_summary(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    s = df.get("Drug_class", pd.Series([pd.NA] * n)).map(bucket_drug_class)
    counts = s.value_counts()

    rows = []
    for c in DRUG_CLASS_ORDER:
        k = int(counts.get(c, 0))
        rows.append({
            "Drug class": c,
            "n": k,
            "%": round(k / n * 100, 2) if n else 0.0
        })
    return pd.DataFrame(rows)

def therapeutic_area_top5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    disease_lists = df[col].apply(clean_split_to_unique_list)

    valid_lists = disease_lists[disease_lists.map(len) > 0]
    denom = len(valid_lists)

    if denom == 0:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    exploded = valid_lists.explode()
    counts = exploded.value_counts().head(15)

    out = counts.reset_index()
    out.columns = ["Therapeutic area", "n"]
    out["%"] = (out["n"] / denom * 100).round(2)
    return out

FOCUS_DISEASE_CLASSES = [
    "diseases of the circulatory system",
    "diseases of the nervous system",
    "neoplasms",
    "endocrine, nutritional and metabolic diseases",
    "infectious diseases",
]

def therapeutic_area_focus5(df: pd.DataFrame) -> pd.DataFrame:
    col = "Disease_class(es)"
    if col not in df.columns:
        return pd.DataFrame(columns=["Therapeutic area", "n", "%"])

    disease_lists = df[col].apply(clean_split_to_unique_list)

    valid_lists = disease_lists[disease_lists.map(len) > 0]
    denom = len(valid_lists)

    if denom == 0:
        return pd.DataFrame(
            [{"Therapeutic area": c, "n": 0, "%": 0.0} for c in FOCUS_DISEASE_CLASSES]
        )

    rows = []
    for c in FOCUS_DISEASE_CLASSES:
        n = int(valid_lists.map(lambda lst: c in lst).sum())
        pct = round(n / denom * 100, 2)
        rows.append({"Therapeutic area": c, "n": n, "%": pct})

    return pd.DataFrame(rows)

for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(f"{name} | Records: {len(df)}")

    print("\nDecision distribution (per application)")
    display(decision_distribution(df))

    print("\nDrug classes (applications)")
    display(drug_class_summary(df))

    print("\nTherapeutic area composition (Top 15)")
    display(therapeutic_area_top5(df))

df_overall = pd.concat(
    [df for name, df in agencies.items() if name in PAR_AGENCIES],
    ignore_index=True
)

print("\n" + "=" * 70)
print("OVERALL (across PAR based agencies)")
print(f"Records: {len(df_overall)}")

print("\nDecision distribution (OVERALL)")
display(decision_distribution(df_overall))

print("\nDrug classes (OVERALL)")
display(drug_class_summary(df_overall))

print("\nTherapeutic area composition (Top 15, OVERALL)")
display(therapeutic_area_top5(df_overall))

print("\nTherapeutic area composition (selected 5 disease classes)")
display(therapeutic_area_focus5(df_overall))

df_nonpar_overall = pd.concat(
    [df for name, df in agencies.items() if name in NON_PAR_AGENCIES],
    ignore_index=True
)

print("\n" + "=" * 70)
print("OVERALL (Non-PAR agencies: FDA + Health Canada)")
print(f"Records: {len(df_nonpar_overall)}")

print("\nDecision distribution (OVERALL, Non-PAR)")
display(decision_distribution(df_nonpar_overall))

print("\nDrug classes (OVERALL, Non-PAR)")
display(drug_class_summary(df_nonpar_overall))

print("\nTherapeutic area composition (Top 5, OVERALL, Non-PAR)")
display(therapeutic_area_top5(df_nonpar_overall))

print("\nTherapeutic area composition (selected 5 disease classes, OVERALL, Non-PAR)")
display(therapeutic_area_focus5(df_nonpar_overall))

# 4. Administration routes and pharmaceutical forms

In [ ]:
# Agency groups
PAR_AGENCIES = {"EMA", "Swissmedic", "TGA"}
API_AGENCIES = {"FDA", "Health Canada"}

# Load data
df_fda = pd.read_csv(FDA_path)
df_health_canada = pd.read_csv(HEALTHCANADA_path)
df_ema = pd.read_csv(EMA_path)
df_swissmedic = pd.read_csv(SWISSMEDIC_path)
df_tga = pd.read_csv(AUSTRALIA_path)

agencies = {
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "TGA": df_tga,
    "FDA": df_fda,
    "Health Canada": df_health_canada,
}

# Placeholders
PLACEHOLDERS = {"not reported", "na", "n/a", "none", ""}

def administration_route_top10(df: pd.DataFrame) -> pd.DataFrame:
    col = "Administration_route"
    if col not in df.columns:
        return pd.DataFrame(columns=["Administration_route", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Administration_route", "n", "%"])

    exploded = (
        s.str.split(";")
        .explode()
        .astype("string")
        .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    exploded = exploded.replace({
        "intravenous": "parenteral",
        "subcutaneous": "parenteral",
        "intramuscular": "parenteral",
    })

    counts = exploded.value_counts().head(10)
    denom = len(exploded)

    out = counts.reset_index()
    out.columns = ["Administration_route", "n"]
    out["%"] = (out["n"] / denom * 100).round(2)
    return out

def pharmaceutical_form_top10(df: pd.DataFrame) -> pd.DataFrame:
    col = "Pharmaceutical_form"
    if col not in df.columns:
        return pd.DataFrame(columns=["Pharmaceutical_form", "n", "%"])

    s = (
        df[col]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
        .dropna()
    )

    if s.empty:
        return pd.DataFrame(columns=["Pharmaceutical_form", "n", "%"])

    exploded = (
        s.str.split(";")
        .explode()
        .astype("string")
        .str.strip()
    )
    exploded = exploded[~exploded.isin(PLACEHOLDERS)].dropna()

    exploded = exploded.replace({
        "tablet": "tablet / capsule",
        "capsule": "tablet / capsule",
        "solution": "solution / injectable",
        "injectable": "solution / injectable",
    })

    counts = exploded.value_counts().head(10)
    denom = len(exploded)

    out = counts.reset_index()
    out.columns = ["Pharmaceutical_form", "n"]
    out["%"] = (out["n"] / denom * 100).round(2)
    return out

def pharmaceutical_form_by_drug_class(df: pd.DataFrame) -> pd.DataFrame:
    required = {"Pharmaceutical_form", "Drug_class"}
    if not required.issubset(df.columns):
        return pd.DataFrame(
            columns=["Drug_class", "Pharmaceutical_form", "n", "%"]
        )

    df = df.copy()

    df["Pharmaceutical_form"] = (
        df["Pharmaceutical_form"]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
    )

    df["Drug_class"] = (
        df["Drug_class"]
        .astype("string")
        .str.lower()
        .str.strip()
        .mask(lambda x: x.isin(PLACEHOLDERS))
    )

    df = df.dropna(subset=["Pharmaceutical_form", "Drug_class"])

    df = df.assign(
        Pharmaceutical_form=df["Pharmaceutical_form"].str.split(";")
    ).explode("Pharmaceutical_form")

    df["Pharmaceutical_form"] = df["Pharmaceutical_form"].str.strip()

    df["Pharmaceutical_form"] = df["Pharmaceutical_form"].replace({
        "tablet": "tablet / capsule",
        "capsule": "tablet / capsule",
        "solution": "solution / injectable",
        "injectable": "solution / injectable",
        "suspension": "suspension",
    })

    counts = (
        df
        .groupby(["Drug_class", "Pharmaceutical_form"])
        .size()
        .reset_index(name="n")
    )

    counts["%"] = (
        counts["n"]
        / counts.groupby("Drug_class")["n"].transform("sum")
        * 100
    ).round(2)

    return counts.sort_values(
        ["Drug_class", "%"], ascending=[True, False]
    )

# Per-agency output
for name, df in agencies.items():
    print("\n" + "=" * 70)
    print(name)

    print("\nAdministration route (Top 10)")
    display(administration_route_top10(df))

    print("\nPharmaceutical form (Top 10, absolute)")
    display(pharmaceutical_form_top10(df))

# Build overall datasets
df_overall_par = pd.concat(
    [df for name, df in agencies.items() if name in PAR_AGENCIES],
    ignore_index=True
).drop_duplicates()

df_overall_api = pd.concat(
    [df for name, df in agencies.items() if name in API_AGENCIES],
    ignore_index=True
).drop_duplicates()

# Overall outputs
print("\n" + "=" * 70)
print("Overall - PAR agencies")
display(administration_route_top10(df_overall_par))
display(pharmaceutical_form_by_drug_class(df_overall_par))

print("\n" + "=" * 70)
print("Overall - FDA & Health Canada")
display(administration_route_top10(df_overall_api))
display(pharmaceutical_form_by_drug_class(df_overall_api))

# Paper summary: top pharmaceutical form per drug class
def top_pharmaceutical_form_per_drug_class(
    df: pd.DataFrame,
    drug_class_patterns: dict
 ) -> pd.DataFrame:

    df_norm = pharmaceutical_form_by_drug_class(df)
    rows = []

    for label, pattern in drug_class_patterns.items():
        subset = df_norm[
            df_norm["Drug_class"].str.contains(pattern, na=False, regex=True)
        ]

        if subset.empty:
            continue

        top_row = subset.sort_values("n", ascending=False).iloc[0]

        rows.append({
            "Drug class": label,
            "Top pharmaceutical form": top_row["Pharmaceutical_form"],
            "n": int(top_row["n"]),
            "% (within drug class)": float(top_row["%"]),
        })

    return pd.DataFrame(rows)

DRUG_CLASS_PATTERNS = {
    "Small molecules": "small molecule",
    "Biologics & advanced therapies": "biologics|peptides|cell and gene",
    "Vaccines": "vaccin|immunological",
}

print("\n" + "=" * 70)
print("Paper summary - FDA & Health Canada")
display(
    top_pharmaceutical_form_per_drug_class(
        df_overall_api, DRUG_CLASS_PATTERNS
    )
)

print("\n" + "=" * 70)
print("Paper summary - PAR agencies")
display(
    top_pharmaceutical_form_per_drug_class(
        df_overall_par, DRUG_CLASS_PATTERNS
    )
)

# 6. Regulatory review durations

In [ ]:
# Helper: compute positive review durations
def compute_review_duration(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds `review_duration_days` = Decision_date - Application_date.
    Keeps only positive durations (> 0 days).
    """
    tmp = df.copy()

    tmp["Application_date"] = pd.to_datetime(
        tmp["Application_date"], errors="coerce", dayfirst=True
    )
    tmp["Decision_date"] = pd.to_datetime(
        tmp["Decision_date"], errors="coerce", dayfirst=True
    )

    tmp["review_duration_days"] = (
        tmp["Decision_date"] - tmp["Application_date"]
    ).dt.days

    tmp = tmp[tmp["review_duration_days"] > 0]

    return tmp.dropna(subset=["review_duration_days"])

def review_duration_median_iqr(df: pd.DataFrame) -> pd.DataFrame:
    """
    Median, Q1, Q3 and IQR of positive review durations (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            median_days="median",
            q1_days=lambda x: x.quantile(0.25),
            q3_days=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    out["iqr_days"] = out["q3_days"] - out["q1_days"]

    return out

def review_duration_min_max(df: pd.DataFrame) -> pd.DataFrame:
    """
    Min and max positive review duration (days),
    stratified by abridged vs non-abridged.
    """
    tmp = compute_review_duration(df)

    tmp["Procedure"] = (
        tmp["Nonclinical_abridged"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"yes": "Abridged", "no": "Non-abridged"})
    )

    tmp = tmp.dropna(subset=["Procedure"])

    out = (
        tmp.groupby("Procedure")["review_duration_days"]
        .agg(
            min_days="min",
            max_days="max",
        )
        .reset_index()
    )

    return out

# Agencies to include; FDA and Health Canada are excluded
agencies_timeline = {
    "EMA": df_ema,
    "Swissmedic": df_swissmedic,
    "TGA": df_australia,
}

for name, df in agencies_timeline.items():
    print("\n" + "=" * 70)
    print(f"{name} - Review timelines (positive durations only)")

    print("\nMedian, Q1, Q3 and IQR (days)")
    display(review_duration_median_iqr(df))

    print("\nMin / Max (days)")
    display(review_duration_min_max(df))

df_overall_timeline = pd.concat(list(agencies_timeline.values()), ignore_index=True)

print("\n" + "=" * 70)
print("OVERALL - Review timelines (positive durations only)")

print("\nMedian, Q1, Q3 and IQR (days)")
display(review_duration_median_iqr(df_overall_timeline))

print("\nMin / Max (days)")
display(review_duration_min_max(df_overall_timeline))

# 7. Relationship between approval activity and disease incidence

In [ ]:
BASE_1995 = Path("./../data/datasets/1995")
APPROVED_DIR = BASE_1995 / "approved"

MAP_PATH = "./../data/Disease_burden_mapping/Mapping_diseases_disease_classes.csv"
GBD_PATH = "./../data/Disease_burden_mapping/Global_disease_burden_statistics_download.csv"

AGENCIES = ["EMA", "FDA", "Health Canada", "TGA", "Swissmedic"]

dfs = []
for agency in AGENCIES:
    fname = f"{agency.replace(' ', '')}.csv"
    path = APPROVED_DIR / fname
    df = pd.read_csv(path)
    df["Agency"] = agency
    dfs.append(df)

APPROVALS_DF = pd.concat(dfs, ignore_index=True)

print(APPROVALS_DF["Agency"].value_counts())

START_YEAR = 1995
END_YEAR = 2023

CANONICAL_CLASSES = [
    "Infectious and parasitic diseases",
    "Neoplasms",
    "Diseases of the blood and blood-forming organs",
    "Endocrine, nutritional, and metabolic diseases",
    "Mental and behavioural disorders",
    "Diseases of the nervous system",
    "Diseases of the eye and adnexa",
    "Diseases of the ear and mastoid process",
    "Diseases of the circulatory system",
    "Diseases of the respiratory system",
    "Diseases of the digestive system",
    "Diseases of the skin",
    "Diseases of the musculoskeletal system and connective tissue",
    "Diseases of the genitourinary system",
    "Pregnancy and childbirth",
    "Congenital malformations and chromosomal abnormalities",
    "Injury, poisoning and certain other consequences of external causes",
    "Other",
]

PLACEHOLDERS = {"not reported", "na", "n/a", "tbd", "none", ""}

def norm_str(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()

def approved_mask_from_decision(decision_series: pd.Series) -> pd.Series:
    dec = decision_series.astype("string").str.lower().str.strip().fillna("")
    return dec.str.contains(r"\b(approved|authorised|authorized)\b", regex=True)

def normalise_disease_class(x) -> str:
    if pd.isna(x):
        return "Other"

    t = str(x).strip()
    if t.lower() in PLACEHOLDERS:
        return "Other"

    fixes = {
        "Diseases of the musculskeletal system and connective tissue":
            "Diseases of the musculoskeletal system and connective tissue",
        "Congenital malformations and chromosal abnormalities":
            "Congenital malformations and chromosomal abnormalities",
        "Injury, poisining and certain other consequences of external causes":
            "Injury, poisoning and certain other consequences of external causes",
    }

    t = fixes.get(t, t)

    if t in CANONICAL_CLASSES:
        return t

    lower_map = {c.lower(): c for c in CANONICAL_CLASSES}
    return lower_map.get(t.lower(), "Other")

def approvals_by_disease_class(approvals_df: pd.DataFrame) -> pd.DataFrame:
    appr = approvals_df.copy()

    appr["Decision_year"] = pd.to_numeric(appr["Decision_year"], errors="coerce")
    appr = appr[appr["Decision_year"].between(START_YEAR, END_YEAR)]

    appr["Disease_class"] = appr["Disease_class(es)"].astype("string").str.split(";")
    appr = appr.explode("Disease_class")
    appr["Disease_class"] = appr["Disease_class"].astype("string").str.strip()
    appr = appr[appr["Disease_class"].notna() & (appr["Disease_class"] != "")]

    appr["Disease_class"] = appr["Disease_class"].map(normalise_disease_class)

    yearly = (
        appr.groupby(["Disease_class", "Decision_year"])
        .size()
        .reset_index(name="approvals_n")
    )

    summary = (
        yearly.groupby("Disease_class")
        .agg(
            mean_approvals_per_year=("approvals_n", "mean"),
            max_approvals_per_year=("approvals_n", "max"),
        )
        .reset_index()
    )

    peak_year = (
        yearly.sort_values(
            ["Disease_class", "approvals_n", "Decision_year"],
            ascending=[True, False, True]
        )
        .drop_duplicates("Disease_class")[["Disease_class", "Decision_year"]]
        .rename(columns={"Decision_year": "peak_year"})
    )

    summary = summary.merge(peak_year, on="Disease_class", how="left")

    total = len(appr)
    share = (
        appr["Disease_class"]
        .value_counts()
        .reindex(CANONICAL_CLASSES, fill_value=0)
        .reset_index(name="approvals_mentions_n")
        .rename(columns={"index": "Disease_class"})
    )

    share["approvals_mentions_pct"] = (
        share["approvals_mentions_n"] / total * 100
    ).round(2) if total else 0.0

    return summary.merge(
        share[["Disease_class", "approvals_mentions_pct"]],
        on="Disease_class",
        how="left"
    )

mapping_raw = pd.read_csv(MAP_PATH, header=None).rename(columns={0: "cause_name"})
mapping_long = mapping_raw.melt(id_vars=["cause_name"], value_name="Disease_class_raw")
mapping_long["cause_name"] = norm_str(mapping_long["cause_name"])
mapping_long["Disease_class"] = mapping_long["Disease_class_raw"].map(normalise_disease_class)
mapping_long = mapping_long.drop_duplicates(["cause_name", "Disease_class"])

gbd = pd.read_csv(GBD_PATH)
gbd["year"] = pd.to_numeric(gbd["year"], errors="coerce")
gbd["cause_name"] = norm_str(gbd["cause_name"])

gbd["measure_name"] = (
    gbd["measure_name"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "incidence": "Incidence",
        "prevalence": "Prevalence",
        "deaths": "Deaths",
        "death": "Deaths",
    })
)

gbd_f = gbd[
    (gbd["measure_name"].isin(["Incidence", "Prevalence", "Deaths"])) &
    (gbd["metric_name"].str.lower() == "rate") &
    (gbd["year"].isin([START_YEAR, END_YEAR]))
]

gbd_mapped = gbd_f.merge(mapping_long, on="cause_name", how="left")
gbd_mapped["Disease_class"] = gbd_mapped["Disease_class"].fillna("Other")

burden = (
    gbd_mapped.groupby(["Disease_class", "measure_name", "year"])["val"]
    .sum()
    .reset_index()
)

burden_wide = (
    burden.pivot_table(
        index=["Disease_class", "measure_name"],
        columns="year",
        values="val"
    )
    .reset_index()
)

burden_wide["pct_change"] = (
    (burden_wide[END_YEAR] - burden_wide[START_YEAR]) /
    burden_wide[START_YEAR] * 100
)

burden_change_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values="pct_change"
    )
    .reindex(CANONICAL_CLASSES)
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_pct_change_{START_YEAR}_{END_YEAR}",
        "Prevalence": f"Prevalence_pct_change_{START_YEAR}_{END_YEAR}",
        "Deaths": f"Deaths_pct_change_{START_YEAR}_{END_YEAR}",
    })
)

burden_end_tbl = (
    burden_wide.pivot_table(
        index="Disease_class",
        columns="measure_name",
        values=END_YEAR
    )
    .reindex(CANONICAL_CLASSES)
    .reset_index()
    .rename(columns={
        "Incidence": f"Incidence_{END_YEAR}",
        "Prevalence": f"Prevalence_{END_YEAR}",
        "Deaths": f"Deaths_{END_YEAR}",
    })
)

burden_class_summary = burden_change_tbl.merge(
    burden_end_tbl, on="Disease_class", how="left"
)

appr_summary_par = approvals_by_disease_class(
    APPROVALS_DF[APPROVALS_DF["Agency"].isin(PAR_AGENCIES)]
)

appr_summary_nonpar = approvals_by_disease_class(
    APPROVALS_DF[APPROVALS_DF["Agency"].isin(NON_PAR_AGENCIES)]
)

combined_par = burden_class_summary.merge(
    appr_summary_par, on="Disease_class", how="left"
)

combined_nonpar = burden_class_summary.merge(
    appr_summary_nonpar, on="Disease_class", how="left"
)

for df in (combined_par, combined_nonpar):
    df["mean_approvals_per_year"] = df["mean_approvals_per_year"].fillna(0)
    df["max_approvals_per_year"] = df["max_approvals_per_year"].fillna(0)
    df["peak_year"] = df["peak_year"].fillna(pd.NA)

candidates = [
    (name, obj)
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame) and "Agency" in obj.columns
 ]

print("DataFrames with an 'Agency' column:")
for name, obj in candidates:
    vc = obj["Agency"].astype("string").str.strip().value_counts()
    print(f"\n{name}:")
    print(vc.head(10))

    if any(a in vc.index.tolist() for a in ["FDA", "Health Canada"]):
        print(">>> FOUND FDA / Health Canada here <<<")

print(f"\n=== PAR agencies: High-burden classes (top 10 by Deaths in {END_YEAR}) ===")
display(
    combined_par.sort_values(f"Deaths_{END_YEAR}", ascending=False).head(10),
)

print("\n=== PAR agencies: Combined table (all disease classes) ===")
display(
    combined_par.set_index("Disease_class").reindex(CANONICAL_CLASSES).reset_index(),
)

print(f"\n=== NON-PAR agencies: High-burden classes (top 10 by Deaths in {END_YEAR}) ===")
display(
    combined_nonpar.sort_values(f"Deaths_{END_YEAR}", ascending=False).head(10),
)

print("\n=== NON-PAR agencies: Combined table (all disease classes) ===")
display(
    combined_nonpar.set_index("Disease_class").reindex(CANONICAL_CLASSES).reset_index(),
)

## Overlapping drug names across agencies (case-insensitive match)

In [ ]:
# Load and clean data
df = pd.read_csv("./../data/datasets/1995/all_decisions/Overall.csv")

df = df[["Non_proprietary_name", "Drug_name", "Dataset"]]

# Exclude Japan / PMDA records
EXCLUDED_DATASETS = {"pmda", "japan"}
df = df[~df["Dataset"].astype(str).str.strip().str.lower().isin(EXCLUDED_DATASETS)]

df["Non_proprietary_name"] = (
    df["Non_proprietary_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Drug_name"] = (
    df["Drug_name"]
    .astype(str)
    .str.strip()
)

df = df.drop_duplicates()

# Wide table: all drug names per INN per agency, then count unique drug names across agencies
grouped = (
    df
    .groupby(["Non_proprietary_name", "Dataset"])["Drug_name"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

pivot = grouped.pivot(
    index="Non_proprietary_name",
    columns="Dataset",
    values="Drug_name"
)

pivot["n_unique_drug_names"] = pivot.apply(
    lambda row: len(set(
        name
        for cell in row.dropna()
        for name in cell
    )),
    axis=1
)

pivot.to_csv(
    "./../output/overlap/drug_names_per_Substance_per_agency.csv"
)

# Number of unique agencies per INN
agency_count = (
    df
    .groupby("Non_proprietary_name")["Dataset"]
    .nunique()
    .rename("n_agencies")
    .reset_index()
)

# Drug name consistency: for each (INN, Drug_name) pair, count in how many agencies this pairing appears
name_agency_map = (
    df
    .groupby(["Non_proprietary_name", "Drug_name"])["Dataset"]
    .nunique()
    .reset_index(name="n_agencies_per_name")
)

# Globally consistent name = the drug name is used by all agencies that report on this INN
name_agency_map = name_agency_map.merge(
    agency_count,
    on="Non_proprietary_name",
    how="left"
)

name_agency_map["is_globally_consistent"] = (
    name_agency_map["n_agencies_per_name"]
    == name_agency_map["n_agencies"]
)

# Summary per INN
summary = (
    name_agency_map
    .groupby("Non_proprietary_name")
    .agg(
        n_agencies=("n_agencies", "first"),
        drug_names=("Drug_name", lambda x: sorted(set(x))),
        has_globally_consistent_name=("is_globally_consistent", "any")
    )
    .reset_index()
)

summary.to_csv(
    "./../output/overlap/drug_name_consistency_summary.csv",
    index=False
)

# Detailed mapping of (INN, Drug_name) pairs with agency counts and consistency flag
name_agency_map.to_csv(
    "./../output/overlap/drug_name_origin_by_agency_count.csv",
    index=False
)

n_total = summary.shape[0]
n_consistent = summary["has_globally_consistent_name"].sum()
percent_consistent = (n_consistent / n_total) * 100

print(f"Total number of substances (INNs): {n_total}")
print(f"Substances with at least one globally consistent drug name: {n_consistent}")
print(f"Proportion with globally consistent name: {percent_consistent:.1f}%")

Number of substances with more than one product name

In [ ]:
(pivot["n_unique_drug_names"] > 1).mean()